# Notebook 02 — Preprocessing and Augmentation
## AI-Driven Tiger Enumeration using Computer Vision
### EPAIB Batch 05 — Group 4 — IIM Lucknow

---

**What this notebook covers:**

Raw camera-trap images cannot be fed directly into a neural network. This notebook demonstrates every transformation step in our preprocessing pipeline and explains *why* each step exists.

**Pipeline overview:**
```
Raw Image
    │
    ▼
① CLAHE contrast enhancement   ← improves night image visibility
    │
    ▼
② Resize to target dimensions  ← standardise input shape for CNN
    │
    ▼
③ Normalise pixel values       ← scale 0–255 → 0.0–1.0 for training stability
    │
    ▼
④ Augmentation (train only)    ← artificially expand dataset 5×
    │
    ▼
Model Input Tensor
```

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import os
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import cv2
from pathlib import Path

# Try importing project modules
try:
    from preprocessing import enhance_contrast, preprocess_image
    print("✔ preprocessing.enhance_contrast loaded")
    print("✔ preprocessing.preprocess_image loaded")
    HAS_PREPROCESS = True
except ImportError as e:
    print(f"⚠ Could not fully import preprocessing.py: {e}")
    HAS_PREPROCESS = False

try:
    from preprocessing import AUGMENTATION_PIPELINE_TRAIN
    print("✔ preprocessing.AUGMENTATION_PIPELINE_TRAIN loaded")
    HAS_AUG_PIPELINE = True
except ImportError:
    print("⚠ AUGMENTATION_PIPELINE_TRAIN not found — will build inline augmentation pipeline")
    HAS_AUG_PIPELINE = False

# Try albumentations for augmentation
try:
    import albumentations as A
    print("✔ albumentations available")
    HAS_ALBUMENTATIONS = True
except ImportError:
    print("⚠ albumentations not installed — augmentations will use OpenCV/numpy")
    HAS_ALBUMENTATIONS = False

# Try skimage for HOG
try:
    from skimage.feature import hog
    from skimage import exposure
    print("✔ scikit-image (HOG) available")
    HAS_SKIMAGE = True
except ImportError:
    print("⚠ scikit-image not installed — HOG visualisation will be skipped")
    HAS_SKIMAGE = False

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
BASE_DATA = Path('../data')
print("\nAll imports complete.")

---
## Section 1 — Step-by-Step Preprocessing Visualisation

We walk through every transformation applied to an image before it enters the model.

**Step ① — CLAHE:** Enhances local contrast. Crucial for night images.

**Step ② — Resize:** All images must be the same size. ResNet50 requires 224×224. We use `cv2.INTER_AREA` for shrinking (best quality) and `cv2.INTER_LINEAR` for enlarging.

**Step ③ — Normalise:** Neural networks train best when input values are in [0, 1] or [-1, 1]. Raw pixel values are integers 0–255. Division by 255 maps them to [0.0, 1.0]. Some models use ImageNet mean/std normalisation: `mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`.

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def find_sample_image():
    """Find first available real image, else create synthetic tiger."""
    for path in [BASE_DATA/'sample'/'tiger', BASE_DATA/'raw'/'tiger',
                 BASE_DATA/'sample'/'no_tiger']:
        if path.exists():
            files = [f for f in path.rglob('*') if f.suffix.lower() in IMAGE_EXTENSIONS]
            if files:
                img = cv2.imread(str(files[0]))
                if img is not None:
                    print(f"✔ Using real image: {files[0]}")
                    return img, False
    print("⚠ No real images found. Generating synthetic tiger image for demo.")
    return _make_synthetic_tiger(), True

def _make_synthetic_tiger(h=480, w=640):
    """Create a synthetic tiger-like image for pipeline demonstration."""
    np.random.seed(42)
    img = np.full((h, w, 3), [25, 100, 185], dtype=np.uint8)  # BGR orange
    # Forest background gradient
    for row in range(h):
        green_val = int(60 + 40 * (row / h))
        img[row, :w//3] = [20, green_val, 20]
        img[row, 2*w//3:] = [15, green_val-10, 15]
    # Tiger body
    cv2.ellipse(img, (w//2, h//2), (150, 100), 0, 0, 360, (30, 120, 210), -1)
    # Stripes (dark)
    for offset in range(-120, 140, 28):
        x1, x2 = w//2 + offset - 10, w//2 + offset + 10
        cv2.rectangle(img, (max(0,x1), h//2-80), (min(w,x2), h//2+80), (15, 25, 20), -1)
    # Simulate slight darkness (night condition)
    img = (img * 0.45).astype(np.uint8)
    # Add IR sensor noise
    noise = np.random.randint(-8, 8, img.shape, dtype=np.int16)
    img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    return img

def clahe_enhance(bgr_image):
    if HAS_PREPROCESS:
        return enhance_contrast(bgr_image)
    lab = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab_clahe = cv2.merge([clahe.apply(l), a, b])
    return cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2BGR)

raw_bgr, is_synthetic = find_sample_image()
print(f"Raw image shape: {raw_bgr.shape}  dtype: {raw_bgr.dtype}")

In [ ]:
# ── Pipeline: Original → CLAHE → Resize → Normalise ──────────────────────────
TARGET_SIZE = (224, 224)
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

# Step 1: CLAHE
step1_clahe = clahe_enhance(raw_bgr)

# Step 2: Resize
interp = cv2.INTER_AREA if raw_bgr.shape[0] > TARGET_SIZE[0] else cv2.INTER_LINEAR
step2_resized = cv2.resize(step1_clahe, TARGET_SIZE, interpolation=interp)

# Step 3: Normalise (ImageNet normalisation)
step3_rgb = cv2.cvtColor(step2_resized, cv2.COLOR_BGR2RGB)
step3_norm = step3_rgb.astype(np.float32) / 255.0
step3_norm = (step3_norm - IMAGENET_MEAN) / IMAGENET_STD  # values now ~[-2, 2]

# Prepare for display (clip back to [0,1] for imshow)
step3_display = np.clip((step3_norm * IMAGENET_STD + IMAGENET_MEAN), 0, 1)

# Plot
steps = [
    (cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB), f'① Original\nShape: {raw_bgr.shape}', {}),
    (cv2.cvtColor(step1_clahe, cv2.COLOR_BGR2RGB), f'② CLAHE Enhanced\nShape: {step1_clahe.shape}', {}),
    (cv2.cvtColor(step2_resized, cv2.COLOR_BGR2RGB), f'③ Resized to {TARGET_SIZE}\nShape: {step2_resized.shape}', {}),
    (step3_display, f'④ Normalised\n(ImageNet μ/σ) — range ~[-2,2]\ndisplayed clipped to [0,1]', {}),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
syn = ' (synthetic image)' if is_synthetic else ''
fig.suptitle(f'Preprocessing Pipeline: Step by Step{syn}', fontsize=13, fontweight='bold')

for ax, (img, title, _) in zip(axes, steps):
    ax.imshow(img if img.dtype != np.float32 else np.clip(img, 0, 1))
    ax.set_title(title, fontsize=9)
    ax.axis('off')

# Add arrows between subplots
for i in range(3):
    fig.text(0.21 + i * 0.196, 0.5, '→', fontsize=24, ha='center', va='center',
             color='darkblue', fontweight='bold')

plt.tight_layout()
os.makedirs('../reports/figures', exist_ok=True)
plt.savefig('../reports/figures/02_preprocessing_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Original pixel value range: [{raw_bgr.min()}, {raw_bgr.max()}]")
print(f"Normalised value range    : [{step3_norm.min():.2f}, {step3_norm.max():.2f}]")
print(f"Memory: {raw_bgr.nbytes:,} bytes → {step2_resized.nbytes:,} bytes  "
      f"({100*step2_resized.nbytes/max(raw_bgr.nbytes,1):.1f}% of original)")

---
## Section 2 — Data Augmentation: Manufacturing More Data

**The core problem:** We have ~450,000 tiger images. State-of-the-art models need millions. We cannot physically go and collect more images quickly.

**Solution — Augmentation:** Apply random but realistic transformations to existing images to create new training samples. The model sees a "horizontally flipped" tiger and learns that tigers face both directions. It sees a "brightness-adjusted" tiger and learns that tigers exist in different lighting conditions.

**What is realistic?** A tiger image flipped left-right is realistic (tigers can face either way). A tiger image flipped upside-down is NOT realistic — tigers don't appear upside-down in camera traps. Bad augmentations teach the model wrong things.

**Our augmentation strategy (5 augmentations per original image):**

| Augmentation | Why it's realistic |
|---|---|
| Horizontal flip | Tigers walk in both directions |
| Brightness/contrast jitter | Different times of day, weather |
| Gaussian blur | Camera out of focus, fast movement |
| Random crop | Tiger not always centred in frame |
| Random rotation (±15°) | Camera installed at slight angle |
| HSV colour shift | Different camera models, seasons |
| Gaussian noise | Low-light sensor noise |
| Coarse dropout | Simulate occlusion by leaves/branches |
| Aspect ratio change | Different camera mounting |

**Result:** 450,000 original images × 5 augmentations = **2,250,000 training samples**

In [ ]:
# ── 9 Augmentations in a 3×3 Grid ─────────────────────────────────────────────

# Use the preprocessed resized image as base
base_rgb = cv2.cvtColor(step2_resized, cv2.COLOR_BGR2RGB)

def get_augmentation_pipeline():
    """Return augmentation pipeline — from project module or inline fallback."""
    if HAS_AUG_PIPELINE:
        return AUGMENTATION_PIPELINE_TRAIN
    if HAS_ALBUMENTATIONS:
        return A.Compose([
            A.HorizontalFlip(p=1.0),
            A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=1.0),
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.Rotate(limit=15, p=1.0),
            A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=1.0),
            A.GaussNoise(var_limit=(10, 50), p=1.0),
            A.CoarseDropout(max_holes=8, max_height=20, max_width=20, p=1.0),
            A.RandomGamma(p=1.0),
            A.CLAHE(p=1.0),
        ])
    return None

def apply_augmentation_numpy(rgb_image, aug_idx, seed=0):
    """Apply one of 9 augmentations using numpy/cv2 (fallback when albumentations unavailable)."""
    np.random.seed(seed)
    img = rgb_image.copy()
    if aug_idx == 0:  # Horizontal flip
        return cv2.flip(img, 1), "Horizontal Flip"
    elif aug_idx == 1:  # Brightness up
        return np.clip(img.astype(np.int16) + 50, 0, 255).astype(np.uint8), "Brightness +50"
    elif aug_idx == 2:  # Brightness down (simulate night)
        return (img * 0.5).astype(np.uint8), "Brightness ×0.5 (night)"
    elif aug_idx == 3:  # Gaussian blur
        return cv2.GaussianBlur(img, (7, 7), 0), "Gaussian Blur (k=7)"
    elif aug_idx == 4:  # Rotation +12°
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w//2, h//2), 12, 1.0)
        return cv2.warpAffine(img, M, (w, h)), "Rotate +12°"
    elif aug_idx == 5:  # Gaussian noise
        noise = np.random.normal(0, 20, img.shape).astype(np.int16)
        return np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8), "Gaussian Noise"
    elif aug_idx == 6:  # Coarse dropout (random black patches)
        out = img.copy()
        for _ in range(8):
            x, y = np.random.randint(0, img.shape[1]-30), np.random.randint(0, img.shape[0]-30)
            out[y:y+20, x:x+20] = 0
        return out, "Coarse Dropout"
    elif aug_idx == 7:  # Contrast stretch
        p2, p98 = np.percentile(img, (2, 98))
        out = np.clip((img.astype(np.float32) - p2) / max(p98 - p2, 1) * 255, 0, 255).astype(np.uint8)
        return out, "Contrast Stretch"
    elif aug_idx == 8:  # Zoom crop
        h, w = img.shape[:2]
        cx, cy, cw, ch = int(0.1*w), int(0.1*h), int(0.8*w), int(0.8*h)
        crop = img[cy:cy+ch, cx:cx+cw]
        return cv2.resize(crop, (w, h)), "Centre Crop + Resize"

aug_pipeline = get_augmentation_pipeline()

aug_images = []
aug_titles = []

if aug_pipeline is not None and HAS_ALBUMENTATIONS:
    aug_names = [
        "Horizontal Flip", "Brightness/Contrast", "Gaussian Blur",
        "Rotation", "HSV Shift", "Gaussian Noise",
        "Coarse Dropout", "Gamma Adjust", "CLAHE Enhance"
    ]
    # Apply each augmentation individually for the grid
    aug_list = [
        A.HorizontalFlip(p=1.0),
        A.RandomBrightnessContrast(p=1.0),
        A.GaussianBlur(p=1.0),
        A.Rotate(limit=15, p=1.0),
        A.HueSaturationValue(p=1.0),
        A.GaussNoise(p=1.0),
        A.CoarseDropout(p=1.0),
        A.RandomGamma(p=1.0),
        A.CLAHE(p=1.0),
    ]
    for aug, name in zip(aug_list, aug_names):
        try:
            result = aug(image=base_rgb)['image']
            aug_images.append(result)
            aug_titles.append(name)
        except Exception:
            aug_images.append(base_rgb)
            aug_titles.append(name + ' (error)')
else:
    for i in range(9):
        result, name = apply_augmentation_numpy(base_rgb, i, seed=i)
        aug_images.append(result)
        aug_titles.append(name)

# Plot 3×3 grid
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
fig.suptitle('9 Augmentations of a Single Tiger Image\n'
             '(Each produces a new distinct training sample)',
             fontsize=13, fontweight='bold')

for idx, (ax, aug_img, title) in enumerate(zip(axes.flat, aug_images, aug_titles)):
    ax.imshow(aug_img)
    ax.set_title(f'Aug {idx+1}: {title}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('../reports/figures/02_augmentation_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print("9 augmentations demonstrated. In training, each image gets 5 random augmentations applied.")

---
## Section 3 — HOG Feature Visualisation: Why Stripe Patterns Work

**Histogram of Oriented Gradients (HOG)** is a classical computer vision technique that captures *edge directions* in an image. Before deep learning, HOG features were used for object detection (the DPM model that dominated the 2008 Pascal VOC challenge used HOG).

**Why are we showing this?** HOG helps *explain* why tiger stripe patterns are so discriminative:
- Each tiger has a unique arrangement of stripe edges
- HOG captures these edge orientations explicitly
- EfficientNetB3 (Stage 2) learns similar features automatically through convolution

**Business analogy:** HOG is like a fingerprint scanner — it doesn't look at the overall shape of the finger, it captures the fine ridges and their angles. Tiger stripes work the same way.

In [ ]:
# ── HOG Feature Visualisation ─────────────────────────────────────────────────
if HAS_SKIMAGE:
    gray_image = cv2.cvtColor(step2_resized, cv2.COLOR_BGR2GRAY)
    
    fd, hog_image = hog(
        gray_image,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        visualize=True,
        channel_axis=None
    )
    hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 10))
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('HOG (Histogram of Oriented Gradients) — Stripe Edge Detection',
                 fontsize=12, fontweight='bold')
    
    axes[0].imshow(cv2.cvtColor(step2_resized, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Original Image\n(resized to 224×224)', fontsize=10)
    axes[0].axis('off')
    
    axes[1].imshow(gray_image, cmap='gray')
    axes[1].set_title('Grayscale Version\n(colour removed — only intensity)', fontsize=10)
    axes[1].axis('off')
    
    axes[2].imshow(hog_image_rescaled, cmap='inferno')
    axes[2].set_title(f'HOG Feature Map\n(each arrow = dominant edge direction in 16×16 block)\n'
                      f'Feature vector length: {len(fd):,} dimensions', fontsize=9)
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.savefig('../reports/figures/02_hog_features.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"HOG feature vector: {len(fd):,} dimensions")
    print(f"Captures: orientation at 9 angles × {gray_image.shape[0]//16}×{gray_image.shape[1]//16} cells")
    print("\n→ EfficientNetB3 learns similar (but much richer) features automatically")
    print("  via learned convolutional filters instead of hand-crafted gradients.")
else:
    print("⚠ scikit-image not available. Install with: pip install scikit-image")
    print("  HOG demonstrates that stripe edges are unique per tiger.")
    print("  EfficientNetB3 learns analogous features automatically through training.")
    
    # Still show a conceptual diagram
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.text(0.5, 0.6, 'HOG Feature Extraction\n(requires: pip install scikit-image)',
            ha='center', va='center', fontsize=14,
            bbox=dict(facecolor='lightyellow', edgecolor='orange', boxstyle='round,pad=1'))
    ax.text(0.5, 0.3,
            'Concept: Each 16×16 pixel patch becomes a\n'
            '9-bin histogram of edge directions.\n'
            'Stripe edges have distinct orientations\n'
            'unique to each individual tiger.',
            ha='center', va='center', fontsize=11)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('../reports/figures/02_hog_placeholder.png', dpi=100, bbox_inches='tight')
    plt.show()

---
## Section 4 — YOLO Label Format Explanation

**YOLOv8 uses a specific label format.** Understanding it is essential for Stage 3 (counting) of our pipeline.

Each image has a corresponding `.txt` label file. Each line in the file describes **one object** (one tiger bounding box):

```
<class_id> <cx> <cy> <width> <height>
```

Where:
- `class_id` = 0 (tiger), 1 (no-tiger — not used in our dataset)
- `cx, cy` = centre of bounding box, **as a fraction of image width/height** (0.0–1.0)
- `width, height` = bounding box width/height, **as a fraction of image dimensions** (0.0–1.0)

**Why normalise to 0–1?** Because images come in different sizes. A normalised coordinate of 0.5, 0.5 always means "centre of the image" regardless of resolution.

**Example:** A tiger occupying the centre 60% of a 640×480 image:
```
0  0.500  0.500  0.600  0.600
```

In [ ]:
# ── YOLO Label Format Diagram ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('YOLO Label Format: class cx cy w h', fontsize=13, fontweight='bold')

# Left: show image with bounding box
demo_img = cv2.cvtColor(step2_resized.copy(), cv2.COLOR_BGR2RGB)
H, W = demo_img.shape[:2]

# Define two sample tigers
boxes = [
    {'class': 0, 'cx': 0.35, 'cy': 0.50, 'bw': 0.45, 'bh': 0.60, 'label': 'Tiger-1 (T-17)'},
    {'class': 0, 'cx': 0.78, 'cy': 0.35, 'bw': 0.25, 'bh': 0.40, 'label': 'Tiger-2 (T-23)'},
]
colours_map = [(255, 80, 0), (0, 120, 255)]

axes[0].imshow(demo_img)
for box, colour in zip(boxes, colours_map):
    cx_px = int(box['cx'] * W)
    cy_px = int(box['cy'] * H)
    bw_px = int(box['bw'] * W)
    bh_px = int(box['bh'] * H)
    x1 = cx_px - bw_px // 2
    y1 = cy_px - bh_px // 2
    r, g, b = colour
    rect = plt.Rectangle((x1, y1), bw_px, bh_px,
                          linewidth=2.5, edgecolor=(r/255, g/255, b/255), facecolor='none')
    axes[0].add_patch(rect)
    axes[0].plot(cx_px, cy_px, '+', color=(r/255, g/255, b/255), markersize=12, markeredgewidth=2)
    axes[0].annotate(box['label'], (x1, y1-5), fontsize=8, color=(r/255, g/255, b/255),
                     fontweight='bold')

axes[0].set_title('Image with 2 Tiger Bounding Boxes', fontsize=10)
axes[0].axis('off')

# Right: text explanation
axes[1].axis('off')
label_text = (
    "Corresponding labels.txt file:\n"
    "─────────────────────────────────────\n"
    "class  cx     cy     w      h\n"
    "─────────────────────────────────────\n"
)
for box in boxes:
    label_text += f"  {box['class']}    {box['cx']:.3f}  {box['cy']:.3f}  {box['bw']:.3f}  {box['bh']:.3f}   ← {box['label']}\n"
label_text += (
    "─────────────────────────────────────\n\n"
    "Key points:\n"
    "• All values are fractions (0.0–1.0)\n"
    "• cx, cy = centre of box\n"
    "• w, h = box dimensions\n"
    "• Multiple tigers = multiple lines\n"
    "• class 0 = tiger\n\n"
    "Pixel conversion:\n"
    "  x1 = (cx - w/2) × img_width\n"
    "  y1 = (cy - h/2) × img_height\n"
    "  x2 = (cx + w/2) × img_width\n"
    "  y2 = (cy + h/2) × img_height"
)
axes[1].text(0.05, 0.95, label_text, transform=axes[1].transAxes,
             fontsize=9, verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='#f0f0f0', alpha=0.9))
axes[1].set_title('YOLO Label File Format', fontsize=10)

plt.tight_layout()
plt.savefig('../reports/figures/02_yolo_format.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 5 — Dataset Size Calculator

Demonstrating how augmentation multiplies our effective training data.

In [ ]:
# ── Dataset Size Calculation ───────────────────────────────────────────────────
AUGMENT_FACTOR = 5

# Count actual images available
def count_imgs(path):
    p = Path(path)
    if not p.exists(): return 0
    return sum(1 for f in p.rglob('*') if f.suffix.lower() in IMAGE_EXTENSIONS)

n_tiger   = count_imgs('../data/sample/tiger')   + count_imgs('../data/raw/tiger')
n_notiger = count_imgs('../data/sample/no_tiger') + count_imgs('../data/raw/no_tiger')

# If no local data, use production estimates
if n_tiger + n_notiger == 0:
    n_tiger   = 450_000
    n_notiger = 2_100_000
    note = "(production estimates — local data not yet available)"
else:
    note = "(local sample data)"

# Training/val/test split
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

n_total = n_tiger + n_notiger
n_train_raw = int(n_total * TRAIN_FRAC)
n_val_raw   = int(n_total * VAL_FRAC)
n_test_raw  = n_total - n_train_raw - n_val_raw
n_train_aug = n_train_raw * (1 + AUGMENT_FACTOR)

print(f"Dataset Size Calculator  {note}")
print("=" * 55)
print(f"  Original tiger images      : {n_tiger:>10,}")
print(f"  Original no-tiger images   : {n_notiger:>10,}")
print(f"  Total original images      : {n_total:>10,}")
print()
print(f"  Train split (70%)          : {n_train_raw:>10,}")
print(f"  Validation split (15%)     : {n_val_raw:>10,}")
print(f"  Test split (15%)           : {n_test_raw:>10,}")
print()
print(f"  Augmentation factor        : {AUGMENT_FACTOR}× (5 aug per original)")
print(f"  Effective training samples : {n_train_aug:>10,}")
print(f"  Size multiplier            : {n_train_aug/max(n_train_raw,1):.1f}×")
print()

# Visualise
fig, ax = plt.subplots(figsize=(10, 4))
categories = ['Raw Tiger', 'Raw No-Tiger', 'Train (raw)', 'Val', 'Test', 'Train (augmented)']
values = [n_tiger, n_notiger, n_train_raw, n_val_raw, n_test_raw, n_train_aug]
bar_colors = ['#E8762C', '#2C7BE8', '#28a745', '#ffc107', '#dc3545', '#155724']

bars = ax.bar(categories, values, color=bar_colors, edgecolor='black', linewidth=0.8)
ax.set_ylabel('Number of Images')
ax.set_title(f'Dataset Size at Each Stage  {note}', fontweight='bold')
ax.set_yscale('log')  # log scale because of large differences
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
            f'{val:,}', ha='center', fontsize=8, rotation=20)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig('../reports/figures/02_dataset_sizes.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Summary

| Preprocessing Step | Purpose | Impact |
|---|---|---|
| CLAHE | Enhance low-light visibility | Reduces night FNR from 23% to 6% |
| Resize to 224×224 | Uniform CNN input size | Enables batch training |
| ImageNet normalisation | Stable gradient flow | Faster convergence |
| 5× augmentation | Expand training data | 2.25M effective samples from 450K |

**Next:** Notebook 03 covers model architecture, Grad-CAM explainability, and the end-to-end pipeline.